# C2 Dataset Inspection and Smartphone Pilot Preparation

This notebook prepares an initial development pilot for **Component 2: Helpfulness-Guided Review Evidence Selection with Minority Preservation**.

The notebook:

1. inspects Amazon Reviews 2023 phone metadata,
2. identifies accessory-filtering issues,
3. evaluates title-keyword filtering,
4. selects phone candidates using structured metadata,
5. extracts review records for an initial smartphone pilot, and
6. saves a clean pilot dataset for later EDA and Function 1 development.

> **Note:** This pilot is for PP1 development only. It is **not** the final research sample.


In [ ]:
from datasets import load_dataset
from itertools import islice
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd

print("C2 environment ready")
print("Pandas version:", pd.__version__)


In [ ]:
PHONE_META_PATH = (
    "hf://datasets/McAuley-Lab/Amazon-Reviews-2023/"
    "raw_meta_Cell_Phones_and_Accessories/*.parquet"
)

PHONE_REVIEW_PATH = (
    "hf://datasets/McAuley-Lab/Amazon-Reviews-2023/"
    "raw/review_categories/Cell_Phones_and_Accessories.jsonl"
)

print("Dataset paths configured.")


## 1. Inspect Phone Metadata

The `Cell_Phones_and_Accessories` collection is inspected first because it contains both actual phones and a large number of accessories. The first step is to verify the available metadata fields and observe the product-category structure.


In [ ]:
phone_meta = load_dataset(
    "parquet",
    data_files=PHONE_META_PATH,
    split="train",
    streaming=True
)

first_phone_product = next(iter(phone_meta))

print("Available metadata fields:")
print(list(first_phone_product.keys()))

metadata_preview = {
    key: first_phone_product.get(key)
    for key in [
        "main_category",
        "title",
        "average_rating",
        "rating_number",
        "categories",
        "parent_asin",
    ]
}

metadata_preview


In [ ]:
PHONE_SAMPLE_SIZE = 2_000

phone_meta = load_dataset(
    "parquet",
    data_files=PHONE_META_PATH,
    split="train",
    streaming=True
)

phone_sample = list(islice(phone_meta, PHONE_SAMPLE_SIZE))
phone_df = pd.DataFrame(phone_sample)

print("Sample size:", len(phone_df))
print("Columns:")
print(phone_df.columns.tolist())

phone_df[["parent_asin", "title", "categories"]].head(10)


In [ ]:
category_counter = Counter()

for categories in phone_df["categories"].dropna():
    if isinstance(categories, list):
        for category in categories:
            category_counter[str(category)] += 1

print("Most common category labels:")
for category, count in category_counter.most_common(30):
    print(f"{count:4d}  {category}")


## 2. Check Whether Title Keywords Are Reliable

A simple title-keyword search is tested only as an exploratory step. Accessory products often include terms such as **"smartphone"** or **"cell phone"** in their titles, so this method is expected to produce false positives.


In [ ]:
phone_terms = [
    "smartphone",
    "cell phone",
    "mobile phone",
    "unlocked phone",
]

possible_phones = phone_df[
    phone_df["title"]
    .fillna("")
    .str.lower()
    .apply(lambda x: any(term in x for term in phone_terms))
]

print("Possible phone products in sample:", len(possible_phones))

possible_phones[
    ["parent_asin", "title", "categories", "rating_number"]
].head(30)


### Initial Dataset Finding

The `Cell_Phones_and_Accessories` metadata collection contains both actual mobile phones and many accessories such as cases, stands, chargers, and holders.

A title-keyword search also produces many false positives because accessory products frequently contain terms such as **"cell phone"** and **"smartphone"** in their titles.

Therefore, the initial product-identification rule uses **structured metadata categories** rather than title keywords. Products containing the exact metadata category `Cell Phones` are treated as phone candidates. A second filtering stage is then used to distinguish smartphones from basic or feature phones where required.


## 3. Expand the Metadata Candidate Pool

The initial 2,000-product sample contains only a small number of products in the exact `Cell Phones` category. The metadata sample is therefore expanded to improve coverage before review extraction.


In [ ]:
EXPANDED_META_SAMPLE_SIZE = 50_000

phone_meta_stream = load_dataset(
    "parquet",
    data_files=PHONE_META_PATH,
    split="train",
    streaming=True
)

expanded_phone_sample = list(
    islice(phone_meta_stream, EXPANDED_META_SAMPLE_SIZE)
)

expanded_phone_df = pd.DataFrame(expanded_phone_sample)

expanded_cell_phone_candidates = expanded_phone_df[
    expanded_phone_df["categories"].apply(
        lambda x: isinstance(x, list) and "Cell Phones" in x
    )
].copy()

print("Expanded metadata sample size:", len(expanded_phone_df))
print("Exact 'Cell Phones' candidates:", len(expanded_cell_phone_candidates))
print(
    "Unique phone products:",
    expanded_cell_phone_candidates["parent_asin"].nunique()
)

display(
    expanded_cell_phone_candidates[
        [
            "parent_asin",
            "title",
            "average_rating",
            "rating_number",
            "categories",
        ]
    ]
    .sort_values("rating_number", ascending=False)
    .head(30)
)


## 4. Inspect Review Schema

The review stream is inspected next. Only fields required for Component 2 are displayed in the preview. Pseudonymous user identifiers and review-image URLs are intentionally excluded from the displayed preview because they are not required for this research component.


In [ ]:
phone_reviews_stream = load_dataset(
    "json",
    data_files=PHONE_REVIEW_PATH,
    split="train",
    streaming=True
)

first_review = next(iter(phone_reviews_stream))

print("Available review fields:")
print(list(first_review.keys()))

review_preview = {
    key: first_review.get(key)
    for key in [
        "rating",
        "title",
        "text",
        "asin",
        "parent_asin",
        "timestamp",
        "helpful_vote",
        "verified_purchase",
    ]
}

review_preview


## 5. Match Reviews to Phone Candidates

The first 2,000,000 records in the review stream are scanned for development purposes. Review collection is capped at 100 reviews per candidate product so that the pilot remains manageable.

These limits are **development settings only** and are not final research sampling rules.


In [ ]:
candidate_asins = set(
    expanded_cell_phone_candidates["parent_asin"].dropna()
)

TARGET_REVIEWS_PER_PRODUCT = 100
MAX_RECORDS_TO_SCAN = 2_000_000

candidate_reviews = []
candidate_counts = defaultdict(int)

phone_reviews_stream = load_dataset(
    "json",
    data_files=PHONE_REVIEW_PATH,
    split="train",
    streaming=True
)

for i, review in enumerate(phone_reviews_stream):

    parent_asin = review.get("parent_asin")

    if parent_asin in candidate_asins:
        if candidate_counts[parent_asin] < TARGET_REVIEWS_PER_PRODUCT:
            candidate_reviews.append(review)
            candidate_counts[parent_asin] += 1

    if i + 1 >= MAX_RECORDS_TO_SCAN:
        print("Reached scan safety limit.")
        break

print("Scanned records:", i + 1)
print("Matched reviews:", len(candidate_reviews))


In [ ]:
counts_df = pd.DataFrame(
    [
        {
            "parent_asin": asin,
            "review_count_collected": count,
        }
        for asin, count in candidate_counts.items()
        if count > 0
    ]
)

counts_df = counts_df.merge(
    expanded_cell_phone_candidates[
        ["parent_asin", "title", "rating_number"]
    ].drop_duplicates("parent_asin"),
    on="parent_asin",
    how="left",
)

counts_df = counts_df.sort_values(
    "review_count_collected",
    ascending=False,
)

print("Products with at least one matched review:", len(counts_df))
display(counts_df.head(30))


### Review Availability Observation

The expanded metadata sample provides a much larger candidate pool than the original 2,000-product sample. This makes it possible to identify several smartphone products with enough review records for an initial PP1 development pilot.

The review-stream scan limit and per-product review cap are temporary development controls. They should not be interpreted as final sampling thresholds.


## 6. Build the Development Smartphone Pilot

The exact `Cell Phones` metadata category contains both smartphones and basic/feature phones. For the PP1 development pilot, a small set of clearly identifiable smartphones is manually verified from the candidate list.

This manual verification is used **only for the development pilot**. A reproducible smartphone-filtering rule should be finalized before the final research experiments.


In [ ]:
SMARTPHONE_PILOT_ASINS = {
    "B09PPWVQSB",  # Samsung Galaxy S20 FE 5G
    "B09JJCXC2M",  # Samsung Galaxy S10e
    "B0BMWTL2S6",  # Samsung Galaxy S22 Ultra
    "B08XRDVWQ2",  # BLU G90 Pro
    "B0BL9TKSBL",  # Moto G Power 2022
    "B07GYY6VTD",  # Moto G6
    "B09HFW8GRB",  # Samsung Galaxy S20 5G
    "B0B8PDLMMS",  # Samsung Galaxy S21 Ultra 5G
    "B08PP9FM9N",  # Apple iPhone 12 Mini
    "B07SDQ483V",  # BLU Vivo XL4
}

smartphone_pilot_reviews = [
    review
    for review in candidate_reviews
    if review.get("parent_asin") in SMARTPHONE_PILOT_ASINS
]

smartphone_pilot_df = pd.DataFrame(smartphone_pilot_reviews)

print("Total smartphone pilot reviews:", len(smartphone_pilot_df))

display(
    smartphone_pilot_df["parent_asin"]
    .value_counts()
    .rename("review_count")
    .to_frame()
)


In [ ]:
pilot_metadata = (
    expanded_cell_phone_candidates[
        ["parent_asin", "title", "average_rating", "rating_number"]
    ]
    .drop_duplicates("parent_asin")
)

pilot_summary = (
    smartphone_pilot_df["parent_asin"]
    .value_counts()
    .rename("review_count")
    .reset_index()
    .merge(
        pilot_metadata,
        on="parent_asin",
        how="left",
    )
    .sort_values("review_count", ascending=False)
)

display(pilot_summary)


### Development Pilot Note

This smartphone pilot is used only to:

- validate the data pipeline,
- perform initial EDA,
- inspect helpful-vote behaviour,
- test the usefulness-prediction workflow, and
- support early PP1 implementation evidence.

It is not the final evaluation dataset. The final research dataset will later include a broader smartphone sample and a laptop sample using documented, reproducible filtering rules.


## 7. Save a Clean Pilot Dataset

Only fields required for Component 2 are saved. Pseudonymous user IDs and image URLs are excluded because they are not required for the planned usefulness modelling, review representation, or evidence-selection experiments.


In [ ]:
PILOT_DIR = Path("../data/pilot")
PILOT_DIR.mkdir(parents=True, exist_ok=True)

smartphone_pilot_path = PILOT_DIR / "smartphone_pilot_raw.parquet"

PILOT_COLUMNS = [
    "rating",
    "title",
    "text",
    "asin",
    "parent_asin",
    "timestamp",
    "helpful_vote",
    "verified_purchase",
]

smartphone_pilot_clean = smartphone_pilot_df[PILOT_COLUMNS].copy()

smartphone_pilot_clean.to_parquet(
    smartphone_pilot_path,
    index=False,
)

print("Saved pilot dataset to:")
print(smartphone_pilot_path)
print("Rows saved:", len(smartphone_pilot_clean))
print("Columns saved:", smartphone_pilot_clean.columns.tolist())
